# Rafeeq · run everything in Colab

1. Add your OpenRouter key: 🔑 **Secrets** (left sidebar) → name `OPENROUTER_API_KEY`, turn on **Notebook access**.
2. Run the cells from the top. Nothing is installed on your own machine.

Models, judge and fallbacks are set in `agentkit.py` (overridable with `AGENT_MODEL`, `AGENT_FALLBACKS`, `JUDGE_MODEL`).

In [ ]:
# 1 · setup: clone, key from Colab Secrets, dependencies
import os, pathlib
if not pathlib.Path("/content/rafeeq-capstone").exists():
    !git clone -q https://github.com/CS-Shahad/rafeeq-capstone.git /content/rafeeq-capstone
%cd /content/rafeeq-capstone
!pip install -q uv
from google.colab import userdata
os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
os.environ["PYTHONUTF8"] = "1"
!uv sync -q && echo "ready" 

In [ ]:
# 2 · offline tests of the defences (no key, no model calls)
!uv run pytest -q

In [ ]:
# 3 · one request end to end (expected: 750 SAR, via compute_fee)
!uv run rafeeq.py ask "What is the fee for an illuminated 3 m² sign?" 

## Evaluation: A = no skills, B = skills loaded on demand
Each configuration is a separate cell, so results are saved even if the other one is interrupted. About 10–20 minutes each.

In [ ]:
!uv run rafeeq.py eval tasks.jsonl --config A --pause 2

In [ ]:
!uv run rafeeq.py eval tasks.jsonl --config B --pause 2

In [ ]:
# table for A and B (→ results/summary.md) + error analysis (→ results/error_analysis.md)
!uv run rafeeq.py analyze

In [ ]:
import pandas as pd
# read every failed reply: the root cause is in the transcript, not in the score
for c in "AB":
    d = pd.read_csv(f"results/eval_{c}.csv")
    for _, r in d[~d.passed].iterrows():
        print(f"\n===== [{c}] {r.id} | {r.failed_graders}\nTOOLS: {r.tools}\nREPLY: {str(r.final)[:700]}")

## Security: the injection tasks with the guard off, then on

In [ ]:
!uv run rafeeq.py redteam

In [ ]:
# download results/ (eval CSVs, summary, error analysis, red team) to commit to the repository
!zip -qr results.zip results artifacts/rafeeq/evals/audit_A.jsonl artifacts/rafeeq/evals/audit_B.jsonl 2>/dev/null; ls -la results
from google.colab import files
files.download("results.zip")

## Live demo (see DEMO.md)
A conversation with approvals: type in the box, answer `y` to approve a write action, empty line to quit.

In [ ]:
!uv run rafeeq.py chat --session demo --reset